In [ ]:
import numpy as np
import os
import sys

sys.path.append('../../..')
path=os.path.abspath(os.path.join(os.getcwd(), "../../../outputs/kuramoto/nnc"))

import torch
import torch.optim.swa_utils as swa_utils
from types import SimpleNamespace

from torch import nn
from torch import profiler
from tqdm import tqdm

import torchsde
from core.utils import objective, adj_lattice, criticalK, order_param,  load_clean_state_dict, sample_kuramoto,loss_compute_kur
from core.policies_kur import  HyperKuramoto,  find_params
from configs.kuramoto_cfg import eval_cfg
import gc
from IPython.display import clear_output

is_cuda = torch.cuda.is_available()
device = 'cuda' if is_cuda else 'cpu'

if not is_cuda:
    print("Warning: CUDA not available; falling back to CPU but this is likely to be very slow.")



In [ ]:

# Ensure Adjacency and Laplacian are ready

%store -r adj_matrix
A_torch = torch.as_tensor(adj_matrix, device=device, dtype=torch.float32)
kcrit, L = criticalK(adj_matrix, eval_cfg.state_size, eval_cfg.scale)
K = 0.01 * kcrit
dist = torch.distributions.Normal(0, 1)

# 2. Evaluation Logic
def run_kuramoto_test(layers, config, A_torch, L_torch):
    # Model Initialization
  
    model = HyperKuramoto(
        config.batch_size, config.state_size, K, A_torch, 
        layers, config.skip, config.strategy, device, eval_cfg.activation
    ).to(device)
    
    ckpt_path = f"{path}/{config.strategy}ema{layers}.pth"
    load_clean_state_dict(model, ckpt_path)
    model.eval()

    # Parameter counting
    p_count = sum(p.numel() for p in model.poly.parameters())
    
    # Statistics Trackers (Welford's Algorithm)
    count, mean, m2 = 0, 0.0, 0.0
    ts_exp = torch.linspace(0, config.t_total, config.t_size).to(device)

    # Inference Block
    with torch.inference_mode():
        pbar = tqdm(range(config.iter), desc=f"Testing L{layers}", leave=False)
        for _ in pbar:
            # Sample initial phases and frequencies
            freqs, ys_init = sample_kuramoto(
                config.state_size, config.batch_size, dist, 
                config.scale, config.scale_kin, device
            )
            model.freqs = freqs
            
            # Neural SDE Integration
            ys_tray = torchsde.sdeint_adjoint(
                model, ys_init, ts_exp, method='reversible_heun', 
                dt=ts_exp[1]-ts_exp[0]
            )
            
            # Cost Calculation: Synchronization Objective + Control Energy
            # terminal_phases = ys_tray[-1, :, 0:config.state_size]
            obj = objective(ys_tray[-1, :, 0:config.state_size], A_torch)
            energy_penalty = config.weight * ys_tray[-1, :, -1]
            
            batch_costs = (obj + energy_penalty).cpu().numpy()

            # Batch Welford Update
            for x in batch_costs:
                count += 1
                delta = x - mean
                mean += delta / count
                m2 += delta * (x - mean)

            # Update Progress Bar with current stats
            if count % config.batch_size == 0:
                current_ste = np.sqrt(m2 / (count - 1)) / np.sqrt(count) if count > 1 else 0
                pbar.set_postfix({"Mean": f"{mean:.4f}", "STE": f"{current_ste:.2e}"})

    final_ste = np.sqrt(m2 / (count - 1)) / np.sqrt(count)
    return mean, final_ste, p_count

# 3. Main Loop
global_results = []


for l in range(1, 5):
    m, s, p_num = run_kuramoto_test(l, eval_cfg, A_torch, L)
    global_results.append({
        "layers": l,
        "mean_cost": m,
        "ste": s,
        "params": p_num
    })
    
    # Explicit cleanup per layer set
    torch.cuda.empty_cache()
    gc.collect()



for res in global_results:
    print(f"{res['layers']:<10} | {res['params']:<12,} | {res['mean_cost']:<12.5f} | {res['ste']:<12.2e}")
